In [3]:
import math
# import streamlit as st

In [9]:
# =================== constant ======================

# ===== 箱型价格($) =====
BIG_BOX_PRICE = 158
MEDIUM_BOX_PRICE = 138
SMALL_BOX_PRICE = 98


# ===== 箱型对应重量(lbs) =====
BIG_BOX_WEIGHT_LIMIT = 55
MEDIUM_BOX_WEIGHT_LIMIT = 45
SMALL_BOX_WEIGHT_LIMIT = 35


# ===== 单边超长阈值(inch) =====
LONGEST_SIDE_LIMIT = 48
SECOND_LONGEST_SIDE_LIMIT = 28


# ===== girth 阈值 (inch) =====
# girth = a + 2(b+h)
OVERSIZE_GIRTH_LIMIT = 130
GIRTH_SURCHARGE_LIMIT = 95


# ===== 超长费用($) =====
INTERNATIONAL_LONG_FEE = 50
DOMESTIC_LONG_FEE = 30


# ===== Oversize费用($) =====
OVERSIZE_FEE = 100


# ===== 不同物流线路 大箱限重(lbs)  =====
INTERNATIONAL_WEIGHT_LIMIT = 55
DOMESTIC_WEIGHT_LIMIT = 50


# =====  不同物流线路 超重费率 =====
INTERNATIONAL_OVERWEIGHT_RATE = 3.6
INTERNATIONAL_OVERWEIGHT_PENALTY = 30

DOMESTIC_OVERWEIGHT_RATE = 1.5
DOMESTIC_OVERWEIGHT_PENALTY = 25






In [4]:
# ================== input validation =================
def get_unit_input():
    while True:
        unit = input("请选择单位（inch/cm）：").strip().lower()

        if unit in ["inch", "cm"]:
            return unit
        else:
            print("❌ 只能输入 inch 或 cm")


def get_dimension_input(name, unit):
    while True:
        try:
            value = float(input(f"请输入{name}（{unit}）："))

            if value <= 0:
                print("❌ 尺寸必须大于 0")
                continue

            return value

        except ValueError:
            print("❌ 请输入有效数字")


def get_international_input():
    while True:
        value = input("是否国际件？(y/n)：").strip().lower()

        if value == "y":
            return True
        elif value == "n":
            return False
        else:
            print("❌ 只能输入 y 或 n")







# ================== 普通output版本的额外费用定价函数 =================

def calculate_extra_fee(length, width, height, is_international, unit):
    
    # 如果是 cm，先换算成 inch
    if unit == "cm":
        length = length / 2.54
        width = width / 2.54
        height = height / 2.54
    
    # 箱子三维进一法到精确到个位数，单位为 inch
    length = math.ceil(length)
    width = math.ceil(width)
    height = math.ceil(height)
    
    # 排序：a=最长边, b=次长边, h=最短边
    dims = sorted([length, width, height], reverse=True)
    a, b, h = dims

    # 体积重，向上取整
    dim_weight = math.ceil((length * width * height) / 139)

    # 超重费
    overweight_fee = 0
    
    if is_international:
        weight_limit = 55
        overweight_rate = 3.6
        penalty = 30
        long_fee = 50
    else:
        weight_limit = 50
        overweight_rate = 1.5
        penalty = 25
        long_fee = 30

    if dim_weight > weight_limit:
        overweight_lbs = dim_weight - weight_limit
        overweight_fee = overweight_lbs * overweight_rate + penalty

    # 超长 / 单边超长 / oversize
    size_fee = 0
    size_fee_name = ""

    girth_value = a + 2 * (b + h)

    if girth_value >= 130: ############  130 
        size_fee = 100
        size_fee_name = "Oversize超长费"
    elif a >= 48:
        size_fee = long_fee
        size_fee_name = "单边超长费"
    elif a < 48 and b >= 28:
        size_fee = long_fee
        size_fee_name = "单边超长费"
    elif girth_value >= 105: ############ 105 
        size_fee = long_fee
        size_fee_name = "超长费"

    total_extra_fee = overweight_fee + size_fee


    

    # ================== 输出展示部分 =================

    
    
    print("\n📦 额外费用计算")
    print(f"体积重：({length} × {width} × {height}) / 139 = {dim_weight} lbs")

    parts = []

    if overweight_fee > 0:
        parts.append(f"{overweight_fee:g}（超重费）")
        print(
            f"超重费：({dim_weight}-{weight_limit}) × "
            f"{overweight_rate} + {penalty} = "
            f"{overweight_fee:g} 美金"
        )
    else:
        print("超重费：0 美金")

    print("\n📏 超长规则检查")

    if a >= 48:
        print(f"❌ 最长边 = {a} inch ≥ 48 inch → 触发单边超长费 {long_fee} 美金")
    else:
        print(f"✅ 最长边 = {a} inch < 48 inch")

    if b >= 28:
        print(f"❌ 次长边 = {b} inch ≥ 28 inch → 触发单边超长费 {long_fee} 美金")
    else:
        print(f"✅ 次长边 = {b} inch < 28 inch")

    if girth_value >= 130:
        print(
            f"❌ Oversize超长："
            f"{a} + 2 × ({b} + {h}) = {girth_value} ≥ 130 "
            f"→ 触发 Oversize超长费 100 美金"
        )
    elif girth_value >= 105:
        print(
            f"❌ 最长边+横截面周长："
            f"{a} + 2 × ({b} + {h}) = {girth_value} ≥ 105 "
            f"→ 触发超长费 {long_fee} 美金"
        )
    else:
        print(
            f"✅ 最长边+横截面周长："
            f"{a} + 2 × ({b} + {h}) = {girth_value} < 105"
        )

    if size_fee > 0:
        parts.append(f"{size_fee:g}（{size_fee_name}）")
        print(f"\n最终收取：{size_fee_name} = {size_fee:g} 美金")
    else:
        print("\n最终收取：超长相关费用 = 0 美金")

    if parts:
        print(f"\n额外费用 = {' + '.join(parts)} = {total_extra_fee:g} 美金")
    else:
        print("\n额外费用 = 0 美金")

    print("\n🐸 宮里帮您节省了30秒的计算时间！")
    print("🧧 发财！发货！红包拿来！")

    return total_extra_fee




In [6]:
# 测试
calculate_extra_fee(40, 24, 22, True, "inch")
print("-----")
calculate_extra_fee(30, 22, 20, False, "inch")
print("-----")
calculate_extra_fee(65, 12, 6, True, "inch")


📦 额外费用计算
体积重：(40 × 24 × 22) / 139 = 152 lbs
超重费：(152-55) × 3.6 + 30 = 379.2 美金

📏 超长规则检查
✅ 最长边 = 40 inch < 48 inch
✅ 次长边 = 24 inch < 28 inch
❌ Oversize超长：40 + 2 × (24 + 22) = 132 ≥ 130 → 触发 Oversize超长费 100 美金

最终收取：Oversize超长费 = 100 美金

额外费用 = 379.2（超重费） + 100（Oversize超长费） = 479.2 美金

🐸 宮里帮您节省了30秒的计算时间！
🧧 发财！发货！红包拿来！
-----

📦 额外费用计算
体积重：(30 × 22 × 20) / 139 = 95 lbs
超重费：(95-50) × 1.5 + 25 = 92.5 美金

📏 超长规则检查
✅ 最长边 = 30 inch < 48 inch
✅ 次长边 = 22 inch < 28 inch
❌ 最长边+横截面周长：30 + 2 × (22 + 20) = 114 ≥ 105 → 触发超长费 30 美金

最终收取：超长费 = 30 美金

额外费用 = 92.5（超重费） + 30（超长费） = 122.5 美金

🐸 宮里帮您节省了30秒的计算时间！
🧧 发财！发货！红包拿来！
-----

📦 额外费用计算
体积重：(65 × 12 × 6) / 139 = 34 lbs
超重费：0 美金

📏 超长规则检查
❌ 最长边 = 65 inch ≥ 48 inch → 触发单边超长费 50 美金
✅ 次长边 = 12 inch < 28 inch
✅ 最长边+横截面周长：65 + 2 × (12 + 6) = 101 < 105

最终收取：单边超长费 = 50 美金

额外费用 = 50（单边超长费） = 50 美金

🐸 宮里帮您节省了30秒的计算时间！
🧧 发财！发货！红包拿来！


50

In [7]:
# ========= User Input =========

unit = get_unit_input()

length = get_dimension_input("长度", unit)
width = get_dimension_input("宽度", unit)
height = get_dimension_input("高度", unit)

is_international = get_international_input()

calculate_extra_fee(
    length,
    width,
    height,
    is_international,
    unit
)

请选择单位（inch/cm）：in
❌ 只能输入 inch 或 cm
请选择单位（inch/cm）：inch
请输入长度（inch）：60
请输入宽度（inch）：30
请输入高度（inch）：20
是否国际件？(y/n)：y

📦 额外费用计算
体积重：(60 × 30 × 20) / 139 = 259 lbs
超重费：(259-55) × 3.6 + 30 = 764.4 美金

📏 超长规则检查
❌ 最长边 = 60 inch ≥ 48 inch → 触发单边超长费 50 美金
❌ 次长边 = 30 inch ≥ 28 inch → 触发单边超长费 50 美金
❌ Oversize超长：60 + 2 × (30 + 20) = 160 ≥ 130 → 触发 Oversize超长费 100 美金

最终收取：Oversize超长费 = 100 美金

额外费用 = 764.4（超重费） + 100（Oversize超长费） = 864.4 美金

🐸 宮里帮您节省了30秒的计算时间！
🧧 发财！发货！红包拿来！


864.4

In [4]:
# 使用 Streamlit 生成网页简易UI界面，出现输入框

st.title("口水蛙‘s 额外费用计算器")

unit = st.radio(
    "请选择尺寸单位",
    ["inch", "cm"]
)

length = st.number_input(f"请输入长度（{unit}）", min_value=0.0)
width = st.number_input(f"请输入宽度（{unit}）", min_value=0.0)
height = st.number_input(f"请输入高度（{unit}）", min_value=0.0)

shipping_type = st.radio(
    "是否国际件？",
    ["国际", "国内"]
)

is_international = shipping_type == "国际"

if st.button("计算额外费用"):

    st.balloons()

    calculate_extra_fee(
        length,
        width,
        height,
        is_international,
        unit
    )

2026-05-15 10:25:36.682 
  command:

    streamlit run /opt/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py [ARGUMENTS]


In [8]:
# python -m streamlit run "Miya's 箱子额外费用计算器.py"
# 复制   在bash上本地试运行

In [10]:
# ====================== 大件/重货优惠 计价函数 =====================




def calculate_large_package_discount_price(length, width, height, is_international):
    
    # 三维进一法
    length = math.ceil(length)
    width = math.ceil(width)
    height = math.ceil(height)

    # 排序：a=最长边, b=次长边, h=最短边
    dims = sorted([length, width, height], reverse=True)
    a, b, h = dims

    # 体积重
    dim_weight = math.ceil((length * width * height) / 139)

    # 判断国际 / 国内超长费
    if is_international:
        long_fee = INTERNATIONAL_LONG_FEE
    else:
        long_fee = DOMESTIC_LONG_FEE

    # girth
    girth_value = a + 2 * (b + h)

    # 超长 / 单边超长 / oversize
    size_fee = 0
    size_fee_name = ""

    if girth_value >= OVERSIZE_GIRTH_LIMIT:
        size_fee = OVERSIZE_FEE
        size_fee_name = "Oversize超长费"

    elif a >= LONGEST_SIDE_LIMIT:
        size_fee = long_fee
        size_fee_name = "单边超长费"

    elif b >= SECOND_LONGEST_SIDE_LIMIT:
        size_fee = long_fee
        size_fee_name = "单边超长费"

    elif girth_value >= GIRTH_SURCHARGE_LIMIT:
        size_fee = long_fee
        size_fee_name = "超长费"

    # ===== 拆箱优惠算法 =====

    best_price = float("inf")

    best_big_count = 0
    best_medium_count = 0
    best_small_count = 0
    best_capacity = 0

    for big_count in range(0, 10):
        for medium_count in range(0, 10):
            for small_count in range(0, 10):

                capacity = (
                    big_count * BIG_BOX_WEIGHT_LIMIT
                    + medium_count * MEDIUM_BOX_WEIGHT_LIMIT
                    + small_count * SMALL_BOX_WEIGHT_LIMIT
                )

                price = (
                    big_count * BIG_BOX_PRICE
                    + medium_count * MEDIUM_BOX_PRICE
                    + small_count * SMALL_BOX_PRICE
                )

                if capacity >= dim_weight and capacity > 0:
                    if price < best_price:
                        best_price = price
                        best_big_count = big_count
                        best_medium_count = medium_count
                        best_small_count = small_count
                        best_capacity = capacity

    total_price = best_price + size_fee

    # ===== 输出 =====

    print("========== 大件 / 重货优惠计算 ==========")

    print(f"箱子尺寸：{length} × {width} × {height} inch")
    print(f"体积重：({length} × {width} × {height}) / 139 = {dim_weight} lbs")
    print(f"最长边 + 横截面周长：{a} + 2 × ({b} + {h}) = {girth_value}")

    print("\n--- 超长费用 ---")

    if size_fee > 0:
        print(f"{size_fee_name}：{size_fee} 美金")
    else:
        print("超长相关费用：0 美金")

    print("\n--- 优惠拆分方式 ---")

    split_parts = []

    if best_big_count > 0:
        print(f"大箱：{BIG_BOX_PRICE} × {best_big_count}")
        split_parts.append(f"{BIG_BOX_PRICE} × {best_big_count}")

    if best_medium_count > 0:
        print(f"中箱：{MEDIUM_BOX_PRICE} × {best_medium_count}")
        split_parts.append(f"{MEDIUM_BOX_PRICE} × {best_medium_count}")

    if best_small_count > 0:
        print(f"小箱：{SMALL_BOX_PRICE} × {best_small_count}")
        split_parts.append(f"{SMALL_BOX_PRICE} × {best_small_count}")

    print(f"可覆盖重量：{best_capacity} lbs")

    if size_fee > 0:
        split_parts.append(f"{size_fee}")

    print(f"\n优惠算法总价 = {' + '.join(split_parts)} = {total_price:g} 美金")

    return {
        "dim_weight": dim_weight,
        "girth_value": girth_value,
        "size_fee": size_fee,
        "size_fee_name": size_fee_name,
        "best_big_count": best_big_count,
        "best_medium_count": best_medium_count,
        "best_small_count": best_small_count,
        "best_capacity": best_capacity,
        "total_price": total_price
    }





In [11]:
calculate_large_package_discount_price(40, 24, 22, True)

========== 大件 / 重货优惠计算 ==========
箱子尺寸：40 × 24 × 22 inch
体积重：(40 × 24 × 22) / 139 = 152 lbs
最长边 + 横截面周长：40 + 2 × (24 + 22) = 132

--- 超长费用 ---
Oversize超长费：100 美金

--- 优惠拆分方式 ---
大箱：158 × 1
小箱：98 × 3
可覆盖重量：160 lbs

优惠算法总价 = 158 × 1 + 98 × 3 + 100 = 552 美金


{'dim_weight': 152,
 'girth_value': 132,
 'size_fee': 100,
 'size_fee_name': 'Oversize超长费',
 'best_big_count': 1,
 'best_medium_count': 0,
 'best_small_count': 3,
 'best_capacity': 160,
 'total_price': 552}